# Module 1 · Session 2 — Guided Lab
## Representing Environmental Observations

### How to use this notebook
This is a **guided learning notebook**, not a code demo. Read every explanation and answer each **Predict / Check your understanding** prompt before running the next code cell.

**Estimated time:** 60–90 minutes.

### By the end you should be able to
- build a labelled environmental dataset;
- reason about dimensions and coordinates;
- select observations across space and time;
- handle missingness and quality;
- aggregate environmental data;
- save and reopen a NetCDF dataset.

## 0. Environment check

We need NumPy for numerical arrays, pandas for dates, and xarray for labelled multidimensional data.

Run the next cell. If an import fails, install the missing package in the Python environment used by this notebook and restart the kernel.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("xarray:", xr.__version__)

## 1. From measurement to environmental observation

A number such as `18.2` is not enough to interpret an environmental measurement scientifically. We normally need context:

**variable + value + unit + space + time + provenance**

For example: land-surface temperature = 18.2 °C, at a known location, on a known date, from a known source.

### Check your understanding
Before continuing, answer in a markdown cell:

**Why would `temperature = 18.2` be insufficient for combining observations from multiple places or sensors?**

**Your answer:**

## 2. Raster thinking

The following NDVI values form a regular 3 × 3 grid. Because values are organised as cells covering space, this is a **raster-style representation**.

A NumPy array stores the numbers efficiently, but it does not yet know what the rows/columns mean geographically.

In [ ]:
ndvi_single_date = np.array([
    [0.72, 0.68, 0.31],
    [0.75, 0.70, 0.28],
    [0.81, 0.74, 0.25]
])

print(ndvi_single_date)
print("shape:", ndvi_single_date.shape)
print("dimensions:", ndvi_single_date.ndim)
print("number of values:", ndvi_single_date.size)

### Predict
If the same 3 × 3 area is observed on **three dates**, what shape should the resulting array have if dimensions are ordered `(time, latitude, longitude)`?

Write your prediction before continuing.

**Prediction:**

## 3. Create a three-date dataset

The first axis below represents time. Each 3 × 3 block is the same area observed on a different date.

In [ ]:
ndvi_data = np.array([
    [[0.72, 0.68, 0.31],
     [0.75, 0.70, 0.28],
     [0.81, 0.74, 0.25]],

    [[0.74, 0.71, 0.35],
     [0.78, 0.73, 0.30],
     [0.84, 0.77, 0.27]],

    [[0.70, 0.67, 0.29],
     [0.73, 0.69, 0.26],
     [0.79, 0.72, 0.23]]
])

temperature_data = np.array([
    [[18.2, 18.5, 19.1],
     [17.9, 18.4, 19.3],
     [17.5, 18.0, 18.8]],

    [[21.1, 21.4, 22.0],
     [20.8, 21.3, 22.2],
     [20.4, 20.9, 21.7]],

    [[24.3, 24.7, 25.5],
     [24.0, 24.5, 25.8],
     [23.6, 24.2, 25.1]]
])

print("NDVI shape:", ndvi_data.shape)
print("Temperature shape:", temperature_data.shape)

### Check
You should see `(3, 3, 3)` for both arrays:

- first `3` → three dates;
- second `3` → three latitude positions;
- third `3` → three longitude positions.

A shape describes **organisation**, but not yet the actual dates or coordinates.

## 4. Dimensions versus coordinates

We will label the axes using:

- `time`
- `latitude`
- `longitude`

and attach actual coordinate values to them.

**Dimension:** an axis along which data are organised.  
**Coordinate:** the labels/values associated with positions on that axis.

In [ ]:
dates = pd.to_datetime(["2026-06-01", "2026-06-15", "2026-07-01"])
latitudes = [51.50, 51.49, 51.48]
longitudes = [-3.20, -3.19, -3.18]

print(dates)

### Predict
What advantage do you expect from selecting `time="2026-06-15"` rather than remembering that 15 June happens to be array index `1`?

Write one sentence.

**Your answer:**

## 5. Build an xarray Dataset

A `DataArray` represents one labelled multidimensional variable. A `Dataset` can contain multiple variables sharing coordinates.

Here NDVI and temperature describe the same places and dates, so they belong naturally in one Dataset.

In [ ]:
ds = xr.Dataset(
    data_vars={
        "ndvi": (
            ["time", "latitude", "longitude"],
            ndvi_data
        ),
        "temperature": (
            ["time", "latitude", "longitude"],
            temperature_data
        )
    },
    coords={
        "time": dates,
        "latitude": latitudes,
        "longitude": longitudes
    },
    attrs={
        "description": "NDVI and temperature observations over time and space"
    }
)

ds

### Read the output
Confirm that you can identify:

1. the three dimensions;
2. the coordinate values;
3. the two data variables;
4. the dataset-level description.

If one of these is unclear, do not continue until you can locate it in the displayed Dataset.

## 6. Add variable metadata

The number `22.2` becomes much more interpretable when we know it is **Land Surface Temperature** measured in `degC`.

Metadata should travel with the data whenever possible.

In [ ]:
ds["ndvi"].attrs = {
    "long_name": "Normalized Difference Vegetation Index",
    "units": "1"
}

ds["temperature"].attrs = {
    "long_name": "Land Surface Temperature",
    "units": "degC"
}

ds

## 7. Select one place and time

### Predict
If we specify **time, latitude and longitude**, how many dimensions should remain in the result?

Write your prediction, then run the selection.

**Prediction:**

In [ ]:
point_observation = ds.sel(
    time="2026-06-15",
    latitude=51.49,
    longitude=-3.18
)

point_observation

### Interpretation
Because every dimension was fixed to one coordinate, the result has no remaining dimensions. It is still a Dataset because it contains both `ndvi` and `temperature`.

**Check:** What are the NDVI and temperature values at this place and time? Write them below with units where appropriate.

**Your answer:**

## 8. Leave time unconstrained: extract a time series

Now select the same spatial location but do **not** specify time.

In [ ]:
ndvi_timeseries = ds["ndvi"].sel(
    latitude=51.49,
    longitude=-3.18
)

ndvi_timeseries

### Check
The only remaining dimension should be `time`.

This illustrates a useful rule:

- fix all dimensions → scalar;
- leave time free → time series;
- fix time only → spatial raster.

## 9. Temporal change

`diff(dim="time")` calculates the difference between consecutive observations.

### Predict
For values `0.28 → 0.30 → 0.26`, what two differences do you expect?

**Prediction:**

In [ ]:
ndvi_change = ndvi_timeseries.diff(dim="time")
ndvi_change

### Interpretation
Three observations produce two intervals. xarray labels each difference using the later time coordinate.

Write one sentence describing the direction of NDVI change in each interval.

**Your interpretation:**

## 10. Spatial aggregation

We now want **one mean NDVI value for each date**.

The original variable has dimensions:

`(time, latitude, longitude)`

To preserve time while summarising the study area, which dimensions should be averaged?

Answer before running the next cell.

**Your answer:**

In [ ]:
mean_ndvi = ds["ndvi"].mean(dim=["latitude", "longitude"])
mean_ndvi

### Check
Only `time` should remain. You have converted a raster time series into an area-level environmental time series.

## 11. Missingness

`NaN` is commonly used for a missing numerical observation.

Create a copy so that the original Dataset remains unchanged.

In [ ]:
missing_ds = ds.copy(deep=True)
missing_ds["ndvi"][1, 1, 2] = np.nan

print("Possible NDVI observations:", missing_ds["ndvi"].size)
print("Non-missing NDVI observations:", int(missing_ds["ndvi"].count()))

### Check
Why are `size` and `count()` now different?

**Important:** missingness describes availability. It does not tell us whether a present observation is scientifically good.

**Your answer:**

## 12. Quality masking

A satellite pixel may contain a numerical value but still be unsuitable—for example because of cloud contamination.

For the first date, define a simple quality mask where `True` means usable and `False` means excluded.

In [ ]:
quality = xr.DataArray(
    [
        [True,  True,  False],
        [True,  False, False],
        [True,  True,  True]
    ],
    dims=["latitude", "longitude"],
    coords={
        "latitude": latitudes,
        "longitude": longitudes
    }
)

first_date_ndvi = ds["ndvi"].sel(time="2026-06-01")
clean_ndvi = first_date_ndvi.where(quality)

print("Original mean:", float(first_date_ndvi.mean()))
print("Quality-controlled mean:", float(clean_ndvi.mean()))
clean_ndvi

### Interpretation
Pixels where the mask is `False` become `NaN`.

The quality-controlled mean may be higher **or** lower than the original mean. Masking does not have a preferred direction; it removes observations according to quality criteria.

### Check
Explain in one sentence why `NaN` and `False` in a quality mask represent different ideas.

**Your answer:**

## 13. Save the Dataset as NetCDF

NetCDF is commonly used for self-describing multidimensional scientific data.

We will save the **full Dataset**, not just a derived mean.

In [ ]:
output_dir = Path("../data")
output_dir.mkdir(exist_ok=True)

file_path = output_dir / "environmental_observations.nc"
ds.to_netcdf(file_path)

print("Saved to:", file_path)

### Troubleshooting
If `to_netcdf()` reports that no suitable backend is available, install a NetCDF backend such as `netCDF4` or `h5netcdf` in the same environment, restart the kernel, and rerun the notebook.

## 14. Reopen and verify

A scientific save step is not complete until we verify that the file can be reopened and still contains the expected structure.

In [ ]:
loaded_ds = xr.open_dataset(file_path)
loaded_ds

### Verification checklist
Confirm that the reopened dataset still contains:

- [ ] `time`, `latitude`, `longitude`
- [ ] `ndvi`
- [ ] `temperature`
- [ ] dataset description
- [ ] variable units and long names

## 15. Final environmental question

Using **only `loaded_ds`**, determine:

> On which date did the study area have the highest mean land-surface temperature, and what was that mean temperature?

We first calculate one spatial mean for each date, then find the maximum and its time coordinate.

In [ ]:
mean_temperature = loaded_ds["temperature"].mean(
    dim=["latitude", "longitude"]
)

max_temperature = mean_temperature.max()
max_date = mean_temperature.idxmax(dim="time")

print("Date with highest mean temperature:", max_date.values)
print("Mean temperature:", round(float(max_temperature.values), 2), "degC")

## 16. Mastery check

Without looking back, can you answer these?

1. What makes an environmental observation more than a number?
2. What is the difference between raster and vector data?
3. Why does a raster need georeferencing information?
4. What does `(12, 4, 100, 150)` mean for `(time, band, y, x)`?
5. What is the difference between a dimension and a coordinate?
6. Why is xarray useful for EO data?
7. What is the difference between missingness and quality?
8. How would you preserve `time` while averaging over space?
9. Why should a NetCDF file be reopened after saving?

If you cannot explain at least 8 of the 9 confidently, review the relevant section before attempting the challenge.